# BIOSTAT 707 Checkpoint 2: predicting in-hospital mortality
Author: Ruiyang Hu

## AI Use Statement
OpenAI Codex assisted with inspecting Checkpoint 1, implementing feature extraction and nested cross-validation, drafting explanations, debugging, and reviewing the course checklist. Automated checks below validate identifiers, folds, prohibited predictors, prediction coverage, and saved-model predictions. Successful report generation demonstrates execution, not clinical validity. AI assistance accelerates implementation but can introduce statistical or clinical errors; retained implausible values and observational measurement patterns require particular caution. This disclosure does not claim independent manual or clinical verification by the author.

This report follows the [official checklist](https://steverozen.github.io/biostat707_2026/checkpoint2-3-checklist.html), retrieved October 17, 2026. Intensive care unit (ICU) stays are the unit of analysis. Area under the receiver operating characteristic curve (AUROC) measures discrimination; it does not establish calibration or clinical utility.

In [ ]:
import os, sys, time, json, platform
from pathlib import Path
if sys.platform == 'win32':
    dll = Path(sys.prefix) / 'Library' / 'bin'
    if dll.exists():
        dll_handle = os.add_dll_directory(str(dll))
        os.environ['PATH'] = str(dll) + os.pathsep + os.environ.get('PATH', '')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, joblib
from IPython.display import display, Markdown
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, SplineTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import roc_auc_score, roc_curve
from threadpoolctl import threadpool_limits
from checkpoint2_support import build_features
thread_limit = threadpool_limits(limits=2)
SEED = 707
np.random.seed(SEED)
START = time.perf_counter()
OUT = Path('output')
OUT.mkdir(exist_ok=True)
def table(df, caption, precision=3):
    display(df.style.format(precision=precision, na_rep='NA').set_caption(caption))
def say(text): display(Markdown(text))
def finish(fig, name):
    fig.tight_layout()
    fig.savefig(OUT / name, dpi=140, bbox_inches='tight')
    plt.show()
    plt.close(fig)

## 2.1 Modeling Table
One row summarizes one ICU stay over the inclusive 0–48-hour window. The loader reproduces Checkpoint 1's identifier validation, stable chronological sorting, and replacement of measurement `-1` with missing values. No Checkpoint 1 output is needed. Suspicious extremes (including negative temperatures, height 431.8 cm, and pH 735) and zeros are retained as in Checkpoint 1; no undocumented unit correction or clinical exclusion threshold is imposed. This is a material limitation, especially for means and extrema.

Only `data/set-a/` and `data/Outcomes-a.txt` are read. Outcomes are joined one-to-one by RecordID; missing outcomes or unequal identifier sets fail explicitly. The prediction time is 48 hours, not admission. Early death or discharge can affect measurement availability, and the source data do not support a rigorously defined alive-at-48-hours landmark cohort here.

In [ ]:
X = build_features(Path('data') / 'set-a')
outcomes = pd.read_csv(Path('data') / 'Outcomes-a.txt').set_index('RecordID')
assert X.index.is_unique and outcomes.index.is_unique
assert len(X) == len(outcomes) == 4000 and set(X.index) == set(outcomes.index)
data = X.join(outcomes[['In-hospital_death', 'SAPS-I', 'SOFA']], validate='one_to_one')
y = data['In-hospital_death'].astype(int)
assert y.isin([0, 1]).all() and y.notna().all()
assert not set(['RecordID', 'Length_of_stay', 'Survival', 'SAPS-I', 'SOFA', 'In-hospital_death']).intersection(X.columns)
data.to_csv(OUT / 'cp2_modeling_table.csv')
cohort = pd.DataFrame({'Stays':[len(y)], 'Deaths':[int(y.sum())], 'Mortality (%)':[100*y.mean()], 'Duplicated IDs':[int(X.index.duplicated().sum())]})
table(cohort, 'Table 1. Modeling cohort: one row per ICU stay')
say(f'Table 1 includes **{len(y):,} stays and {y.sum():,} deaths ({100*y.mean():.2f}%)**. Deaths are a minority, motivating stratification; AUROC alone will not establish usefulness in this imbalanced population.')

## 2.2 Predictors
Source abbreviations: ALP, alkaline phosphatase; ALT, alanine aminotransferase; AST, aspartate aminotransferase; BUN, blood urea nitrogen; GCS, Glasgow Coma Scale; HR, heart rate; HCT, hematocrit; WBC, white blood cell count; SysABP/DiasABP/MAP, invasive systolic/diastolic/mean arterial pressure; NISysABP/NIDiasABP/NIMAP, noninvasive counterparts; FiO2, inspired oxygen fraction; PaO2/PaCO2, arterial oxygen/carbon dioxide partial pressures; SaO2, arterial oxygen saturation; HCO3, bicarbonate; Na/K/Mg, sodium/potassium/magnesium; Temp, temperature; MechVent, mechanical ventilation recording. AUROC, CV (cross-validation), L2 (squared-coefficient) penalty, and SD (sample standard deviation) are used below.

Static age, sex coding (Gender), height, and ICU type use their first nonmissing value. Time-varying parameters use last, minimum, maximum, mean, and nonmissing measurement count. Last values summarize the end of the observation window; extrema summarize instability and means summarize overall burden. Counts and never-measured indicators capture observation processes, which may reflect severity and local care practices. They must not be interpreted causally. MechVent contains only recorded ones, so its four constant value summaries are omitted and its count retained; absent records do not prove absence of ventilation.

There is no prevalence-based variable exclusion threshold. Redundant zero-variance transformed columns are removed within each training fold (variance threshold = 0). One missingness flag per source parameter is available before imputation; redundant flags for its four summaries are avoided. ICUType and Gender are one-hot encoded with missing as a separate level and an omitted reference category. Numerical predictors are median-imputed within training folds; medians are robust to extremes and avoid inventing extreme values, but single imputation understates missing-data uncertainty. No outcome-driven feature screening is used.

In [ ]:
categorical = ['Gender', 'ICUType']
numeric = [c for c in X if c not in categorical]
groups = []
for label, cols, calculation, reason in [
    ('Static', [c for c in X if '_' not in c], 'First nonmissing value', 'Baseline case mix and ICU context'),
    ('Last', [c for c in X if c.endswith('_last')], 'Last nonmissing observation; stable source order breaks time ties', 'State at end of window'),
    ('Minimum/maximum', [c for c in X if c.endswith(('_min','_max'))], 'Within-stay extrema', 'Physiological instability'),
    ('Mean', [c for c in X if c.endswith('_mean')], 'Arithmetic mean of available observations', 'Average physiological burden'),
    ('Frequency', [c for c in X if c.endswith('_count')], 'Number of nonmissing observations, zero if absent', 'Monitoring/treatment intensity'),
    ('Missingness', [c for c in X if c.endswith('_missing')], '1 if corresponding value summary is missing', 'Informative test ordering')]:
    groups.append({'Group':label, 'N':len(cols), 'Calculation':calculation, 'Motivation':reason, 'Predictors':', '.join(cols)})
summary = pd.DataFrame(groups)
assert summary.N.sum() == X.shape[1]
table(summary, 'Table 2. Complete predictor inventory before fold-specific encoding and constant removal')
summary.to_csv(OUT / 'cp2_predictor_summary.csv', index=False)
say(f'Table 2 enumerates **{X.shape[1]} input predictors**. Encoding and removal of constant columns change the fitted design dimension by fold. RecordID, Length_of_stay, Survival, SAPS-I and SOFA are excluded categorically, without numerical thresholds; no other source variable is dropped. Highly correlated summaries increase complexity, motivating ridge shrinkage.')

## 2.3 Cross-Validation Design
Five stratified outer folds (seed 707) are materialized once and reused for all models and benchmarks. Each stay is predicted exactly once. Three stratified inner folds, generated only from each outer training subset, tune hyperparameters by negative log loss. AUROC is the evaluation metric. Every imputer, encoder, scaler, spline knot estimator, and constant-column filter is inside a Pipeline and refit in every inner training split. Feature definitions and the nonlinear model's clinical subset are prespecified without examining outcome associations.

After outer evaluation, each model is tuned again by three-fold cross-validation on all set-a and refit. These fits support interpretation and saving, not additional unbiased evaluation. Selecting a model family using outer results introduces remaining selection optimism; untouched external data in Checkpoint 3 are needed. Fold standard deviations are descriptive, not confidence intervals, because training sets overlap. Small grids and two computational threads bound runtime.

In [ ]:
outer = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(X, y))
fold_id = np.zeros(len(y), dtype=int)
for k, (train, test) in enumerate(outer, 1):
    assert not set(train).intersection(test)
    fold_id[test] = k
fold_table = pd.DataFrame({'Fold':fold_id, 'Death':y.to_numpy()}).groupby('Fold').Death.agg(Stays='size', Deaths='sum', Mortality='mean')
table(fold_table, 'Table 3. Shared held-out fold composition')
say('Table 3 checks the common evaluation denominators and event distribution. Stratification balances the death rate without sharing outcomes or fitted preprocessing across the training/test boundary.')
def num_pipe(splines=False):
    steps = [('impute', SimpleImputer(strategy='median', keep_empty_features=True))]
    if splines:
        steps += [('spline', SplineTransformer(n_knots=4, degree=3, knots='quantile', include_bias=False, extrapolation='linear'))]
    return Pipeline(steps + [('scale', StandardScaler())])
cat_pipe = Pipeline([('impute', SimpleImputer(strategy='constant', fill_value=-999, keep_empty_features=True)), ('encode', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))])
def preprocessing(nums, spline=False, extras=None):
    transforms = [('num', num_pipe(spline), nums), ('cat', cat_pipe, categorical)]
    if extras:
        transforms.append(('extra', num_pipe(), extras))
    return ColumnTransformer(transforms, remainder='drop', verbose_feature_names_out=True)
def logistic(pre):
    return Pipeline([('pre', pre), ('constant', VarianceThreshold()), ('model', LogisticRegression(solver='lbfgs', max_iter=3000, random_state=SEED))])
gam_cont = ['Age', 'Weight_last', 'HR_mean', 'Temp_mean', 'SysABP_mean', 'RespRate_mean', 'GCS_min', 'BUN_last', 'Creatinine_last', 'Na_last', 'K_last', 'Glucose_last', 'HCO3_last', 'WBC_last', 'Platelets_last']
gam_missing = ['Weight_last_missing', 'BUN_last_missing', 'Creatinine_last_missing', 'HCO3_last_missing']
models = {
    'Ridge': logistic(preprocessing(numeric)),
    'Additive spline logistic': logistic(preprocessing(gam_cont, True, gam_missing)),
    'Gradient boosting': Pipeline([('pre', preprocessing(numeric)), ('constant', VarianceThreshold()), ('model', HistGradientBoostingClassifier(max_iter=150, early_stopping=False, random_state=SEED))])
}
grids = {'Ridge': {'model__C':[0.01, 0.1, 1.0]},
         'Additive spline logistic': {'model__C':[0.01, 0.1, 1.0]},
         'Gradient boosting': {'model__learning_rate':[0.05, 0.1], 'model__max_leaf_nodes':[7, 15], 'model__l2_regularization':[1.0]}}
oof = pd.DataFrame({'RecordID':X.index, 'In-hospital_death':y.to_numpy(), 'fold':fold_id})
scores, tuning, fitted = {}, [], {}
for name, model in models.items():
    preds, aucs = np.full(len(y), np.nan), []
    for k, (train, test) in enumerate(outer, 1):
        search = GridSearchCV(clone(model), grids[name], scoring='neg_log_loss', cv=StratifiedKFold(3, shuffle=True, random_state=SEED+k), n_jobs=1, error_score='raise')
        search.fit(X.iloc[train], y.iloc[train])
        preds[test] = search.predict_proba(X.iloc[test])[:,1]
        aucs.append(roc_auc_score(y.iloc[test], preds[test]))
        tuning.append({'Model':name, 'Fold':k, 'Parameters':str(search.best_params_), 'Inner log loss':-search.best_score_})
        print(name, 'fold', k, 'AUROC', round(aucs[-1],4), flush=True)
        with (OUT / 'cp2_progress.txt').open('a') as progress:
            progress.write(f'{name} fold {k}: AUROC {aucs[-1]:.4f}\n')
    assert np.isfinite(preds).all()
    oof[name], scores[name] = preds, aucs
    full_search = GridSearchCV(clone(model), grids[name], scoring='neg_log_loss', cv=StratifiedKFold(3, shuffle=True, random_state=SEED), n_jobs=1, error_score='raise')
    full_search.fit(X, y)
    fitted[name] = full_search.best_estimator_
    tuning.append({'Model':name, 'Fold':'Full refit', 'Parameters':str(full_search.best_params_), 'Inner log loss':-full_search.best_score_})
tuning = pd.DataFrame(tuning)
tuning.to_csv(OUT / 'cp2_tuning.csv', index=False)

## 2.4 Regularized Logistic Regression
Ridge (L2) penalization shrinks correlated summary coefficients together rather than arbitrarily selecting one representative. The inverse penalty strength C is selected from 0.01, 0.1 and 1 by inner log loss. No LASSO variable selection or selection stability claim is made. For interpretation, all transformed columns—including one-hot categories—are standardized to unit standard deviation in the equivalent coefficient representation. Coefficients describe conditional associations per one standard deviation, not causal effects. Correlated predictors can redistribute effects across coefficients.

In [ ]:
def fold_results(name, caption):
    table(pd.DataFrame({'Fold':range(1,6), 'Held-out AUROC':scores[name]}), caption)
    say(f'{name} has mean held-out AUROC {np.mean(scores[name]):.3f}, with fold SD {np.std(scores[name], ddof=1):.3f}.')
fold_results('Ridge', 'Table 4. Ridge logistic regression outer-fold discrimination')
ridge = fitted['Ridge']
names = ridge['pre'].get_feature_names_out()[ridge['constant'].get_support()]
design = ridge['constant'].transform(ridge['pre'].transform(X))
coef = pd.Series(ridge['model'].coef_[0] * design.std(axis=0), index=names)
coef.to_csv(OUT / 'cp2_ridge_coefficients.csv', header=['Log odds per SD'])
fig, ax = plt.subplots(figsize=(9, max(10, len(coef)*0.15)))
coef.sort_values().plot.barh(ax=ax, fontsize=6)
ax.axvline(0, color='black', lw=.7)
ax.set(title='Figure 1. Ridge coefficients on a common one-SD scale', xlabel='Conditional log odds per 1 SD')
finish(fig, 'cp2_ridge_coefficients.png')
top = coef.abs().nlargest(5).index.tolist()
say('Figure 1 displays every retained coefficient; the largest absolute standardized associations are ' + ', '.join(top) + '. Missingness and monitoring coefficients may represent clinical workflow. Coefficient magnitudes are sensitive to retained implausible values and collinearity, so they should not be read as a ranking of causal importance.')
table(tuning.loc[tuning.Model.eq('Ridge')], 'Table 5. Ridge tuning choices and full-data refit')
say('Table 5 reports selected C values and inner log loss. The full-data choice is used for the coefficient plot; outer-fold predictions retain their own independently selected settings.')

## 2.5 Nonlinear Logistic Regression
The second model is an additive logistic model with cubic B-spline smooths (a logistic generalized additive model, GAM). Each continuous variable has its own cubic basis, four training-fold quantile knots, and linear extrapolation outside the boundary knots. There are no interactions. Ridge shrinkage on standardized basis coefficients controls flexibility; this is a basis-coefficient penalty, rather than a derivative-based smoothing penalty or a restricted cubic spline fit.

Fifteen continuous predictors represent age/body size, circulation, respiration, neurologic status, kidney function, electrolytes, metabolic status and blood counts. Gender and ICUType are categorical terms (17 clinical inputs total); four prespecified missingness flags supplement these inputs. Selection is clinical and fixed before fitting, not based on Model 1's full-data coefficients. Other model families retain the full feature set, so comparisons assess both representation and model class.

Effects below are differences in model log odds relative to the observed median of each plotted predictor, with other inputs held at a common reference profile. Additivity makes the contrast independent of the other values. The plots cover the 5th–95th observed percentiles to avoid sparse tails. Ninety-five percent pointwise percentile bands come from 100 patient bootstrap refits of the entire preprocessing/model pipeline with the selected full-data C fixed. They include preprocessing uncertainty but exclude hyperparameter selection uncertainty, are not simultaneous bands, and do not test nonlinearity formally.

In [ ]:
fold_results('Additive spline logistic', 'Table 6. Additive spline logistic model outer-fold discrimination')
table(pd.DataFrame({'Continuous smooth terms':gam_cont + ['']*(2), 'Additional clinical terms':['']*15 + categorical}), 'Table 7. Prespecified nonlinear-model clinical inputs')
say('Table 7 prioritizes clinically recognizable organ-system summaries rather than screening on observed mortality. Four missingness flags listed in the code supplement these clinical inputs.')
gam = fitted['Additive spline logistic']
plot_vars = ['Age', 'GCS_min', 'BUN_last']
reference = X.median().to_frame().T
for c in categorical: reference[c] = X[c].mode().iloc[0]
profiles, grids_x, effects = [], [], []
for col in plot_vars:
    xx = np.linspace(*X[col].quantile([.05,.95]), 60)
    profile = pd.concat([reference]*len(xx), ignore_index=True)
    profile[col] = xx
    profiles.append(profile)
    grids_x.append(xx)
    effects.append(gam.decision_function(profile) - gam.decision_function(reference)[0])
rng = np.random.default_rng(SEED)
boot = np.zeros((100, 3, 60))
for b in range(100):
    ix = rng.integers(0, len(y), len(y))
    bm = clone(gam).fit(X.iloc[ix], y.iloc[ix])
    base = bm.decision_function(reference)[0]
    for j, profile in enumerate(profiles): boot[b,j] = bm.decision_function(profile) - base
fig, axes = plt.subplots(1,3,figsize=(14,4))
effect_rows = []
for j, (ax,col,xx,eff) in enumerate(zip(axes,plot_vars,grids_x,effects)):
    low, high = np.quantile(boot[:,j,:], [.025,.975], axis=0)
    ax.plot(xx,eff)
    ax.fill_between(xx,low,high,alpha=.25)
    ax.axhline(0,color='black',lw=.6)
    ax.set(title=f'{col}; reference {reference[col].iloc[0]:.1f}', xlabel=col, ylabel='Log odds relative to median')
    effect_rows.extend({'Predictor':col,'Value':v,'Effect':e,'Lower':l,'Upper':h} for v,e,l,h in zip(xx,eff,low,high))
fig.suptitle('Figure 2. Additive partial effects and 95% pointwise bootstrap bands')
finish(fig, 'cp2_partial_effects.png')
pd.DataFrame(effect_rows).to_csv(OUT / 'cp2_partial_effects.csv', index=False)
for col,xx,eff in zip(plot_vars,grids_x,effects):
    straight = eff[0] + (eff[-1]-eff[0])*(xx-xx[0])/(xx[-1]-xx[0])
    deviation = np.max(np.abs(eff-straight))
    direction = 'increases' if eff[-1] > eff[0] else 'decreases'
    clinical = {'Age':'Increasing age can reflect lower physiological reserve; any reversal requires caution about selection and adjustment.', 'GCS_min':'Lower Glasgow Coma Scale scores can reflect neurologic impairment or sedation; sedation is an important noncausal explanation.', 'BUN_last':'Elevated blood urea nitrogen can reflect renal dysfunction, catabolism or volume depletion; confounding and retained extremes remain possible.'}[col]
    say(f'Figure 2 ({col}): fitted log odds {direction} by {eff[-1]-eff[0]:.2f} from the 5th to 95th percentile. The maximum deviation from the endpoint straight line is {deviation:.2f} log-odds units, indicating ' + ('visible curvature' if deviation > .1 else 'a largely linear pattern on this interval') + f'. {clinical} Curvature is descriptive; the bands do not establish a statistically significant nonlinear effect.')

## 2.6 Tree Ensemble Model
Histogram gradient boosting fits 150 sequential trees. The grid crosses learning rates {0.05, 0.1} and maximum leaf counts {7, 15}, with L2 regularization 1 and default minimum leaf size 20. Early stopping is disabled, so no validation set is drawn from the held-out fold. The same fold-fitted median imputation, category encoding, scaling and constant removal are used as for ridge; scaling is unnecessary for tree splits but ensures consistent preprocessing. Counts and flags preserve missingness information after imputation. The model can capture nonlinearities and interactions, at the cost of reduced transparency.

In [ ]:
fold_results('Gradient boosting', 'Table 8. Gradient boosting outer-fold discrimination')
table(tuning.loc[tuning.Model.eq('Gradient boosting')], 'Table 9. Selected gradient boosting settings')
say('Table 9 shows the training-only tuning decisions. Differences among folds indicate sensitivity to training composition; none of these settings was chosen using the corresponding held-out outcomes.')

## 2.7 Model Comparison and Final Model Selection
Simplified Acute Physiology Score I (SAPS-I) and Sequential Organ Failure Assessment (SOFA) are evaluated directly as ordered risk scores: higher scores imply higher risk. No score-to-probability calibration is needed for AUROC, and they never enter a learned model. Negative benchmark values are treated as unavailable. The primary five-way comparison uses the common subset with both scores available, retaining the original fold assignments; learned models are still trained using all outer-training stays. Full-cohort learned-model AUROC is also reported to expose any change in denominator. Missing benchmark scores are not imputed from held-out data.

In [ ]:
bench = data[['SAPS-I','SOFA']].mask(data[['SAPS-I','SOFA']].lt(0))
common = bench.notna().all(axis=1).to_numpy()
coverage = pd.DataFrame({'Missing':bench.isna().sum(), 'Available':bench.notna().sum()})
table(coverage, 'Table 10. Benchmark availability')
say(f'Table 10 implies a common comparison cohort of {common.sum():,} stays and {y.iloc[np.flatnonzero(common)].sum():,} deaths. Exclusion of missing scores may select a different severity mix; results apply to this complete-score cohort.')
for name in bench:
    oof[name] = bench[name].to_numpy()
oof['benchmark_complete'] = common
comparison = {}
for name in list(models) + list(bench):
    values = []
    for k in range(1,6):
        mask = common & (fold_id == k)
        assert y.iloc[np.flatnonzero(mask)].nunique() == 2
        values.append(roc_auc_score(y.to_numpy()[mask], oof[name].to_numpy()[mask]))
    comparison[name] = values + [np.mean(values), np.std(values, ddof=1)]
comparison = pd.DataFrame(comparison, index=[f'Fold {k}' for k in range(1,6)]+['Mean','SD']).T
table(comparison, 'Table 11. Primary five-way AUROC comparison on identical complete-score stays')
comparison.to_csv(OUT / 'cp2_comparison.csv')
full_comparison = pd.DataFrame({name:values+[np.mean(values),np.std(values,ddof=1)] for name,values in scores.items()}, index=[f'Fold {k}' for k in range(1,6)]+['Mean','SD']).T
table(full_comparison, 'Table 12. Learned-model AUROC on all set-a stays')
say('Table 12 uses all stays rather than excluding unavailable benchmarks; compare its means with Table 11 to assess the practical impact of the benchmark missingness restriction.')
fig, ax = plt.subplots(figsize=(7,6))
for name in comparison.index:
    yy = y.to_numpy()[common]
    pp = oof[name].to_numpy()[common]
    fpr,tpr,_ = roc_curve(yy,pp)
    ax.plot(fpr,tpr,label=f'{name} (pooled AUC {roc_auc_score(yy,pp):.3f})')
ax.plot([0,1],[0,1],'k--',lw=.7)
ax.set(title='Figure 3. Five-way out-of-fold ROC comparison',xlabel='False positive rate',ylabel='True positive rate')
ax.legend(fontsize=8)
finish(fig, 'cp2_roc.png')
best = full_comparison['Mean'].idxmax()
say(f'Figure 3 compares ranking across thresholds on the same stays. Pooled out-of-fold AUROC can differ from mean fold AUROC because it also compares predictions across separately fitted folds. The highest full-cohort mean AUROC is from **{best}** ({full_comparison.loc[best,"Mean"]:.3f}).')
for benchmark in bench:
    diff = comparison.loc[best].iloc[:5].to_numpy() - comparison.loc[benchmark].iloc[:5].to_numpy()
    delta, sd = diff.mean(), diff.std(ddof=1)
    say(f'Against {benchmark}, {best} has a mean paired fold AUROC difference of {delta:+.3f} (SD of paired differences {sd:.3f}); its fold SD is {comparison.loc[best,"SD"]:.3f}, versus {comparison.loc[benchmark,"SD"]:.3f} for the benchmark. The gain is ' + ('larger' if delta > max(sd, comparison.loc[best,'SD'],comparison.loc[benchmark,'SD']) else 'not larger') + ' than these descriptive measures of fold variation. This is not a formal significance test or proof of clinical benefit.')
# Prespecified practical rule: prefer ridge if within 0.01 AUROC of best; otherwise prefer
# additive logistic if within 0.01; otherwise select the highest mean model.
means = full_comparison['Mean']
chosen = next((name for name in ['Ridge','Additive spline logistic'] if means[best]-means[name] <= .01), best)
say(f'The final model is **{chosen}**. The decision rule favors ridge, then additive logistic, when within 0.01 AUROC of the highest mean; otherwise it selects the highest mean. The 0.01 margin is a pragmatic complexity preference, not an established clinical equivalence margin. Ridge is easiest to deploy and explain; an additive model permits smooth effects; boosting adds interactions and complexity. The chosen model is {means[best]-means[chosen]:.3f} below the highest mean AUROC. All full-data fits were tuned and refit on set-a only. External discrimination, calibration and utility remain untested.')
final = fitted[chosen]
joblib.dump(final, OUT / 'cp2_final_model.joblib')
oof.to_csv(OUT / 'cp2_oof_predictions.csv', index=False)
metadata = {'selected_model':chosen, 'seed':SEED, 'predictors':X.columns.tolist(), 'best_parameters':final.get_params()['model'].get_params(), 'python':platform.python_version(), 'sklearn':sklearn.__version__, 'numpy':np.__version__, 'pandas':pd.__version__, 'selection_rule':'Ridge then additive if within 0.01 of highest mean full-cohort outer AUROC', 'feature_builder':'checkpoint2_support.build_features', 'benchmark_complete_n':int(common.sum())}
(OUT / 'cp2_model_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
loaded = joblib.load(OUT / 'cp2_final_model.joblib')
np.testing.assert_allclose(loaded.predict_proba(X.iloc[:20]), final.predict_proba(X.iloc[:20]))
assert oof.RecordID.is_unique and len(oof)==len(y)
assert oof[list(models)].notna().all().all()
assert np.bincount(fold_id)[1:].sum()==len(y)
for filename in ['cp2_final_model.joblib','cp2_oof_predictions.csv','cp2_comparison.csv','cp2_predictor_summary.csv','cp2_ridge_coefficients.png','cp2_partial_effects.png','cp2_roc.png']:
    assert (OUT / filename).is_file()
elapsed = time.perf_counter()-START
metadata['analysis_seconds'] = elapsed
(OUT / 'cp2_model_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
say(f'Automated verification passed: unique stays, aligned outcomes, common folds, complete learned-model out-of-fold predictions, required artifacts, and identical probabilities after saving/loading the fitted pipeline. Analysis runtime was {elapsed/60:.1f} minutes. Python {platform.python_version()}, scikit-learn {sklearn.__version__}, pandas {pd.__version__}.')

### Limitations and reproducibility review
This is retrospective prediction from a single source. Retained implausible observations can distort feature summaries; missingness and measurement frequencies may encode treatment, monitoring, and early discharge/death. For example, MechVent_count measures recorded ventilation activity rather than a causal effect of ventilation. The first 48 hours include clinical interventions and may be truncated by outcomes. A prospective landmark design and clinical data review are needed before deployment. Median imputation and fixed missingness flags simplify incompletely observed physiology; bootstrap bands condition on the selected hyperparameter. High-dimensional correlated summaries can reduce coefficient stability even under ridge shrinkage.

Nested cross-validation protects held-out predictions from preprocessing and tuning leakage within each model family. Model-family selection still uses set-a results, and neither apparent discrimination nor these observational effects imply calibrated probabilities or clinical benefit. No set-b or set-c files have been read. Checkpoint 3 must load the saved pipeline and reuse the feature builder, reindexing columns to `cp2_model_metadata.json`; it must not refit preprocessing or tune on external data.

The source notebook, feature builder, Pixi manifest/lock, and this rendered report form the reproducible submission. All outputs, including the patient-level modeling table and predictions, remain under `output/`; only the HTML report is allowed through `.gitignore`. Checkpoint 1 is preserved. The user's instruction prohibits automatic pushing. Independent author review and Git submission remain outside this report's automated checks; additional execution checks are documented in README.md.